# 第 5 天挑战 —— 公司小册子 + 西班牙语翻译

## 练习目标

在 Week 1 Day 5「公司小册子」流程上再加一步：先用模型生成英文营销小册子，再**流式**把小册子翻译成西班牙语。

## 和本课的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|------------------|
| 抓取网页链接/正文 | `fetch_website_links` / `fetch_website_contents` |
| 用 JSON 结构化输出挑相关链接 | `response_format={"type": "json_object"}` |
| Chat Completions 生成小册子 | `brochure_system_prompt` + `get_brochure_user_prompt` |
| 流式展示翻译结果 | `stream=True` + `update_display` |

## 怎么跑

1. 确保同目录有可用的 `modified_scrapper`，以及 `.env` 里配置了 `OPENAI_API_KEY`
2. 从上到下运行单元格
3. 最后一格对 OpenAI 官网生成小册子并流式输出西班牙语译文


In [ ]:
# ========== 导入：后面抓网页、调 API、展示 Markdown 都要用 ==========

# 标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 标准库 json：把模型返回的 JSON 字符串解析成 Python dict
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display（流式刷新同一块区域）
from IPython.display import Markdown, display, update_display
# 从本目录的 modified_scrapper 导入：抓页面链接列表、抓页面正文
from modified_scrapper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [ ]:
# ========== 初始化：加载密钥、校验格式、创建客户端与模型常量 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API Key（字符串名必须保持 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：有值、以 sk-proj- 开头、长度大于 10 —— 只是启发式，不是正式鉴权
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    # 成功提示文案保持英文原样（依赖程序判断的输出字符串不改）
    print("API key looks good so far")
else:
    # 失败提示文案保持英文原样
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# 本练习选用的模型 id（字符串必须原样保留）
MODEL = 'gpt-5-nano'
# 创建默认 OpenAI 客户端（会自动从环境变量读 OPENAI_API_KEY）
openai = OpenAI()


In [ ]:
# ========== System Prompt：教模型「从链接列表里挑适合写进小册子的页面」 ==========

# 发给模型的 system 指令必须保留英文：改译会改变模型行为
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
# ========== User Prompt 构造：把目标站 URL + 抓到的链接列表拼给模型 ==========

def get_links_user_prompt(url):
    # 先写固定英文说明（prompt 正文保留英文），再把链接列表拼到后面
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    # 调用抓取函数：拿到该站页面上的链接列表
    links = fetch_website_links(url)
    # 把每条链接用换行拼进 user prompt
    user_prompt += "\n".join(links)
    # 返回完整 user 消息文本
    return user_prompt


In [ ]:
# ========== 选相关链接：一次 Chat Completions，强制 JSON Object 响应 ==========

def select_relevant_links(url):
    # 进度打印：当前 URL 与所用模型名
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    # 调用 Chat Completions：system 定规则，user 放链接列表
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        # 要求返回合法 JSON 对象（不是自由文本）
        response_format={"type": "json_object"}
    )
    # 取出助手消息正文（应是一段 JSON 字符串）
    result = response.choices[0].message.content
    # 解析 JSON → Python dict，形状预期含 "links" 列表
    links = json.loads(result)
    # 打印挑中的链接条数
    print(f"Found {len(links['links'])} relevant links")
    # 把结构化结果交给后续抓正文
    return links


In [ ]:
# ========== 汇总素材：落地页正文 + 相关链接页正文，拼成一大段 Markdown ==========

def fetch_page_and_all_relevant_links(url):
    # 抓首页/落地页正文
    contents = fetch_website_contents(url)
    # 让模型挑出相关链接（About / Careers 等）
    relevant_links = select_relevant_links(url)
    # 先写入落地页标题与正文
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    # 逐个相关链接：写小标题，再抓该 URL 正文并追加
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    # 返回给「写小册子」用的长文本素材
    return result


In [ ]:
# ========== System Prompt：根据多页内容写短小册子（面向客户/投资人/求职者） ==========

# prompt 字符串保持英文原样
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [ ]:
# ========== User Prompt：公司名 + 截断后的网站素材，交给模型写小册子 ==========

def get_brochure_user_prompt(company_name, url):
    # 英文 user 说明：公司名 + 要求用 markdown 写短册子（勿改译 prompt）
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    # 追加抓到的落地页与相关页正文
    user_prompt += fetch_page_and_all_relevant_links(url)
    # 截断到 5000 字符，避免 prompt 过长超上下文/烧配额
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [ ]:
# ========== 挑战核心：先生成英文小册子，再流式翻译成西班牙语并刷新显示 ==========

def stream_brochure(company_name, url):
    # 第一次调用：非流式生成英文小册子（模型 id 字符串保持原样）
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ]
    )
    # 取出完整英文小册子正文
    result = response.choices[0].message.content

    # 第二次调用：把小册子翻译成西班牙语；stream=True 边生成边收
    # 注意：翻译指令字符串必须保留英文原样
    stream = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": "Translate the following marketing brochure to Spanish: " + result}
          ],
        stream=True
    )
    # 累积已收到的译文片段
    response = ""
    # 先放一个空的 Markdown 显示句柄，后面用同一 display_id 刷新
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式 chunk：拼 delta，刷新笔记本里的 Markdown
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 运行挑战：对 OpenAI 官网生成小册子并流式输出西班牙语译文 ==========

# 公司名与 URL 保持原样；会触发抓取 + 两次模型调用（可能较慢、有费用）
stream_brochure("OpenAI", "https://openai.com/")
